# Core Method: Metadata-Driven Secure Prompting (MDSP)

**Metadata-Driven Secure Prompting (MDSP)** is a privacy-preserving GenAI workflow where users generate analytical code and insights by sharing only safe dataset metadata while keeping raw data local and protected.

**The Six-Step MDSP Workflow**


1.   **Keep raw data local:** Do not upload sensitive or proprietary datasets directly into an LLM.
2.   **Extract safe metadata:** Identify schema, column names, data types, missing values, aggregate statistics, and general value ranges.
3.	**Remove risky metadata:** Exclude unique identifiers, rare categories, free-text fields, exact timestamps, and values that may expose individuals, devices, or organizations.
4.	**Prompt using metadata only:** Use the metadata summary to ask the LLM to generate Python analysis code.
5.	**Run code locally:** Execute the generated code in a controlled environment such as Google Colab, GitHub Codespaces, or a local Python environment.
6.	**Validate results:** Review the code, assumptions, analysis logic, outputs, and privacy implications before accepting results.





# **STEP 1: Keep raw data local**
> Do not upload sensitive or proprietary datasets directly into an LLM.

In [ ]:
FILE_PATH = "../data/california_housing_combined.csv"

# **STEP 2. Extract safe metadata**
> Identify schema, column names, data types, missing values, aggregate statistics, and general value ranges.

In [ ]:
#Code generated by Gemini
!pip install pandas
import pandas as pd
from pathlib import Path
from urllib.parse import urlparse
import sys

# ==========================================
# CONFIGURATION: Change only the value below
# ==========================================
DATA_SOURCE = r"../data/california_housing_combined.csv"

def main():
    print("LOCAL REVIEW ONLY: Do not copy or upload this output to a GenAI system until the metadata has been reviewed and sanitized.\n")

    if not DATA_SOURCE or DATA_SOURCE == r"<LOCAL_PATH_OR_DIRECT_CSV_URL>":
        print("Error: Please provide a valid local path or URL in the DATA_SOURCE variable.")
        return

    # Determine source type
    parsed_url = urlparse(DATA_SOURCE)
    is_url = bool(parsed_url.scheme in ('http', 'https'))

    if is_url:
        if "/blob/" in DATA_SOURCE:
            print("Error: GitHub /blob/ URL detected. Please open the file on GitHub, click the 'Raw' button, and use that URL instead.")
            return

    try:
        # Load dataset
        mydata = pd.read_csv(DATA_SOURCE)

        if mydata.empty or mydata.shape[1] == 0:
            print("Error: The dataset contains no data or no columns.")
            return

    except Exception as e:
        print(f"Error loading dataset: The file could not be accessed or is not a valid CSV.")
        return

    # 1. Dataset Dimensions
    print(f"Dataset Dimensions: {mydata.shape[0]} rows, {mydata.shape[1]} columns\n")

    # 2. Numbered Column List
    print("Numbered Column List:")
    for i, col in enumerate(mydata.columns, 1):
        print(f"{i}. {col}")

    print("\nDataFrame Information:")
    mydata.info()

    # 3. Missing Data Summary
    print("\nMissing Data Summary:")
    missing_data = pd.DataFrame({
        'data_type': mydata.dtypes,
        'non_null_count': mydata.count(),
        'missing_count': mydata.isnull().sum(),
        'missing_percentage': (mydata.isnull().sum() / len(mydata) * 100).round(2),
        'unique_count': mydata.nunique()
    })
    print(missing_data)

    # 4. Duplicate Rows
    dup_count = mydata.duplicated().sum()
    print(f"\nDuplicate-row count: {dup_count}")
    print(f"Duplicate-row percentage: {(dup_count / len(mydata) * 100):.2f}%")

    # 5. Numerical Summary
    num_cols = mydata.select_dtypes(include=['number'])
    if not num_cols.empty:
        print("\nNumerical Summary:")
        print(num_cols.describe().loc[['count', 'mean', 'std', '25%', '50%', '75%']].round(2))
    else:
        print("\nNo numerical columns detected.")

    # 6. Categorical Summary
    cat_cols = mydata.select_dtypes(exclude=['number'])
    if not cat_cols.empty:
        print("\nCategorical Data Summary:")
        print(missing_data.loc[cat_cols.columns])

    # 7. Create Metadata Review File
    output_path = Path.cwd() / "LOCAL_ONLY_mdsp_metadata_review1.csv"
    if output_path.exists():
        print(f"\nWarning: Replacing existing file at {output_path}")

    metadata_review = pd.DataFrame({
        'original_column': mydata.columns,
        'default_alias': [f"column_{i:03d}" for i in range(1, len(mydata.columns) + 1)],
        'data_type': mydata.dtypes.values,
        'non_null_count': mydata.count().values,
        'missing_count': mydata.isnull().sum().values,
        'missing_percentage': (mydata.isnull().sum() / len(mydata) * 100).round(2).values,
        'unique_count': mydata.nunique().values,
        'student_pii_decision': 'Unsure',
        'risk_type': '',
        'recommended_action': 'Review',
        'safe_to_share': 'No',
        'student_notes': ''
    })

    metadata_review.to_csv(output_path, index=False)

    print(f"\n{'='*20}")
    print("NEXT MDSP STEP")
    print(f"{'='*20}")
    print("1. Open LOCAL_ONLY_mdsp_metadata_review.csv locally.")
    print("2. Review every original column name.")
    print("3. Identify direct identifiers, quasi-identifiers, sensitive attributes,")
    print("   confidential fields, and free-text fields.")
    print("4. Review or replace each default alias.")
    print("5. Select an appropriate recommended action.")
    print("6. Change safe_to_share to Yes only for metadata that is safe to disclose.")
    print("7. Do not upload this local review file to a GenAI system because it still")
    print("   contains the original column names.")
    print("8. Create a separate sanitized metadata file before prompting a GenAI model.")

if __name__ == "__main__":
    main()

# **STEP 3: Remove risky metadata**
> Exclude unique identifiers, rare categories, free-text fields, exact timestamps, and values that may expose individuals, devices, or organizations.

## **Privacy Assessment of This Dataset**

Based on the metadata review, this dataset appears to contain **housing-area aggregate information** rather than direct personal records.

No obvious direct identifiers were identified, such as:

* names;
* email addresses;
* telephone numbers;
* account numbers;
* street addresses;
* household identifiers;
* Social Security numbers;
* free-text comments.

However, the absence of obvious direct identifiers does not automatically make a dataset risk-free. Geographic fields and financial or demographic aggregates may still require protection, particularly when combined with other data.

### Preliminary Field Classification

| Column               | Likely privacy risk             | Recommended treatment                                                                                       |
| -------------------- | ------------------------------- | ----------------------------------------------------------------------------------------------------------- |
| `longitude`          | Geographic quasi-identifier     | Use an alias or generalize the location. Do not share raw coordinate values.                                |
| `latitude`           | Geographic quasi-identifier     | Use an alias or generalize the location. Do not share raw coordinate values.                                |
| `housing_median_age` | Low-risk aggregate              | Generally safe to include as metadata. Do not share individual row values when the dataset is confidential. |
| `total_rooms`        | Low-risk aggregate              | Generally safe to include as metadata.                                                                      |
| `total_bedrooms`     | Low-risk aggregate              | Generally safe to include as metadata.                                                                      |
| `population`         | Low-risk aggregate              | Generally safe to include as metadata.                                                                      |
| `households`         | Low-risk aggregate              | Generally safe to include as metadata.                                                                      |
| `median_income`      | Potentially sensitive aggregate | Usually safe to include as metadata, but do not share row-level values.                                     |
| `median_house_value` | Potentially sensitive aggregate | Usually safe to include as metadata, but do not share row-level values.                                     |

### Important Caution

The `longitude` and `latitude` fields are not direct identifiers by themselves, but exact coordinate pairs may reveal a specific:

* neighborhood;
* census block;
* property;
* facility;
* device location;
* or organization.

For this reason, exact coordinate values should remain local. Only privacy-safe metadata, generalized location information, or approved semantic aliases should be shared with a GenAI system.

Suggested aliases include:

| Original column      | Suggested alias       |
| -------------------- | --------------------- |
| `longitude`          | `geo_longitude`       |
| `latitude`           | `geo_latitude`        |
| `housing_median_age` | `housing_age_median`  |
| `total_rooms`        | `room_count_total`    |
| `total_bedrooms`     | `bedroom_count_total` |
| `population`         | `population_count`    |
| `households`         | `household_count`     |
| `median_income`      | `income_median`       |
| `median_house_value` | `house_value_median`  |

> **MDSP caution:** Metadata is safer than raw data, but metadata is not automatically safe. Review column names, geographic information, rare categories, free-text fields, identifiers, and sensitive aggregates before sharing any metadata with a GenAI system.

This assessment is specific to the current housing dataset and should not be applied automatically to other datasets. Each new dataset requires a separate privacy review.


## **Suggested shareable metadata**
Dataset metadata approved for GenAI sharing:

Dataset dimensions:
- Rows: 20,000
- Columns: 9

Schema:
1. longitude
   - Data type: float64
   - Non-null count: 20,000

2. latitude
   - Data type: float64
   - Non-null count: 20,000

3. housing_median_age
   - Data type: float64
   - Non-null count: 20,000

4. total_rooms
   - Data type: float64
   - Non-null count: 20,000

5. total_bedrooms
   - Data type: float64
   - Non-null count: 20,000

6. population
   - Data type: float64
   - Non-null count: 20,000

7. households
   - Data type: float64
   - Non-null count: 20,000

8. median_income
   - Data type: float64
   - Non-null count: 20,000

9. median_house_value
   - Data type: float64
   - Non-null count: 20,000

Privacy restrictions:
- Raw records are not provided.
- Exact longitude and latitude values are not provided.
- Numerical distributions, ranges, outliers, and individual values are withheld.
- The dataset must remain local.
- Generated code should operate on a local pandas DataFrame named `mydata`.
- All assumptions must be verified locally before analysis.

# **STEP 4: Prompt using metadata only**
> Use the metadata summary to ask the LLM to generate Python analysis code.

# **STEP 5: Run code locally**
> Execute the generated code in a controlled environment such as Google Colab, GitHub Codespaces, or a local Python environment.

# **STEP 6: Validate results**
> Review the code, assumptions, analysis logic, outputs, and privacy implications before accepting results.